### Load features

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

df = pd.read_parquet(PROCESSED_DIR / "features_nbhd_day.parquet")
df["date"] = pd.to_datetime(df["date"])
print(df.shape)

(691408, 46)


### Build citywide daily collisions and surge label

In [8]:
city = (
    df.groupby("date", as_index=False)
      .agg(city_collisions=("collisions", "sum"))
      .sort_values("date")
)

# Top 10% threshold over training period (we will compute on train split only later)
city["surge_label"] = 0  # temp placeholder
city.head()

,date,city_collisions,surge_label
0,2014-01-08,207,0
1,2014-01-09,222,0
2,2014-01-10,155,0
3,2014-01-11,117,0
4,2014-01-12,66,0


### Join surge label back to nbhd rows

In [9]:
df = df.merge(city[["date","city_collisions"]], on="date", how="left")
print(df[["date","city_collisions"]].head())

        date  city_collisions
0 2014-01-08              207
1 2014-01-09              222
2 2014-01-10              155
3 2014-01-11              117
4 2014-01-12               66


### Time split and threshold on train only

In [10]:
# Simple time split
split_date = pd.Timestamp("2024-01-01")  # you can adjust
train = df[df["date"] < split_date].copy()
test  = df[df["date"] >= split_date].copy()

# Compute surge threshold on train city totals
train_city = train.groupby("date", as_index=False).agg(city_collisions=("collisions","sum"))
thr = np.quantile(train_city["city_collisions"], 0.90)

print("Surge threshold (train 90th pct):", thr)

# Label each row by whether its date is a surge day
train["y"] = (train["city_collisions"] >= thr).astype(int)
test["y"]  = (test["city_collisions"] >= thr).astype(int)

train["y"].value_counts(), test["y"].value_counts()

Surge threshold (train 90th pct): 208.0


(y
 0    517450
 1     58460
 Name: count, dtype: int64,
 y
 0    108862
 1      6636
 Name: count, dtype: int64)

### Train LightGBM as baseline

In [11]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import train_test_split

# Features: exclude identifiers + target + ALL current-day collision counts
drop_cols = {
    "y", "date", "area_id", "area_name", "AREA_ID", "AREA_NAME", "geometry",
    "NEIGHBOURHOOD_158", "NEIGHBOURHOOD_140",
    "collisions", "city_collisions",  # <-- FIX: Dropping the leak
    "ksi_collisions", "ksi_fatal_collisions", "ksi_serious_collisions"  # <-- FIX: Dropping severity leaks
}

X_cols = [c for c in train.columns if c not in drop_cols]

X_train = train[X_cols]
y_train = train["y"]
X_test = test[X_cols]
y_test = test["y"]

model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

p_test = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, p_test))
print("PR-AUC:", average_precision_score(y_test, p_test))
print("Brier:", brier_score_loss(y_test, p_test))

[LightGBM] [Info] Number of positive: 58460, number of negative: 517450
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026312 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1818
[LightGBM] [Info] Number of data points in the train set: 575910, number of used features: 37
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.101509 -> initscore=-2.180570
[LightGBM] [Info] Start training from score -2.180570
ROC-AUC: 0.7844204992392722
PR-AUC: 0.18389851771505258
Brier: 0.05914428850645002


### Save predictions

In [12]:
pred = test[["date","nbhd_id","collisions","city_collisions"]].copy()
pred["surge_proba"] = p_test

out_path = PROCESSED_DIR / "pred_citywide_surge.parquet"
pred.to_parquet(out_path, index=False)
print("Saved:", out_path)

Saved: C:\code\pyspark-playground\Covercheck-Toronto\data\processed\pred_citywide_surge.parquet
